# 03 Models

Runs from the README, each in its own section:

| Run | Method | Status |
|---|---|---|
| Baseline | Most popular beers in train | done |
| B | BPR | done |
| C | B + home-country bonus | to do |
| D | BPR on Nordic countries | done |
| E | BPR on a matching non-Nordic sample | done |

Every run is scored with the same `evaluate()` on validation and test users.

In [ ]:
import os
from pathlib import Path

# Run from the project folder so `src` can be imported
if Path.cwd().name == "notebooks":
    os.chdir("..")

## Setup (shared by all runs)

In [ ]:
import pandas as pd

from src.metrics import evaluate
from src.models import popular_beers, recommend_popular
from src.split import load_split

train, history, hidden = load_split()

# Coverage is measured against every beer the models can recommend
n_beers = train["beer_id"].nunique()
GROUPS = ["validation", "test"]
results = []

In [ ]:
def score_run(run_name, recommendations):
    """Evaluate a run on validation and test users and add it to the results."""
    for group in GROUPS:
        group_hidden = hidden[hidden["group"] == group]
        scores = evaluate(recommendations, group_hidden, n_beers)
        results.append({"run": run_name, "group": group, **scores})
    return pd.DataFrame(results).query("run == @run_name")

## Baseline: most popular beers

Rank beers by number of ratings from users outside Norway. Every user gets the same list, minus beers they already rated.

In [ ]:
ranking = popular_beers(train)
baseline_recommendations = recommend_popular(ranking, history)
score_run("baseline", baseline_recommendations)

The 10 most popular beers:

In [ ]:
from src.data import load_beers

beer_info = load_beers().set_index("beer_id")[["beer_name", "style", "brewery_name"]]
beer_info.loc[ranking[:10]]

## Run B: BPR trained on all other countries

BPR learns which beers tend to be rated by the same users. It is trained on `train` plus the 5 history ratings of each Norwegian user.

Pick the model size (`factors`) on validation users only, then score the best one like the baseline.

In [ ]:
from src.models import recommend_bpr

validation_hidden = hidden[hidden["group"] == "validation"]
bpr_tuning = pd.DataFrame({
    factors: evaluate(recommend_bpr(train, history, factors=factors), validation_hidden, n_beers)
    for factors in [32, 64, 128]
}).T
bpr_tuning

In [ ]:
best_factors = bpr_tuning["ndcg@10"].idxmax()
bpr_recommendations = recommend_bpr(train, history, factors=best_factors)
score_run("B", bpr_recommendations)

## Run D: BPR trained on Nordic countries only

Same BPR as run B (same `factors`), but `train` only has users from Sweden, Denmark and Finland. The Norwegian history is still included. Run E will use the same number of ratings from non-Nordic countries, so D and E only differ in where the data comes from (H3).

In [ ]:
NORDIC = ["Sweden", "Denmark", "Finland"]

nordic_train = train[train["user_country"].isin(NORDIC)]
print(f"{len(nordic_train):,} ratings from {nordic_train['user_id'].nunique():,} Nordic users")

nordic_recommendations = recommend_bpr(nordic_train, history, factors=best_factors)
score_run("D", nordic_recommendations)

## Run E: BPR trained on a matching non-Nordic sample

Randomly sample ratings from users outside Sweden, Denmark, Finland and Norway, without replacement. Use exactly as many source ratings as Run D, the same Norwegian history and the same BPR factors. The fixed seed makes the sample repeatable. Comparing D and E tests H3.

In [ ]:
from src.split import SEED, TARGET_COUNTRY

# Run E uses countries outside both the Nordic source countries and Norway.
excluded_countries = NORDIC + [TARGET_COUNTRY]
is_excluded_country = train["user_country"].isin(excluded_countries)
non_nordic_train = train[is_excluded_country == False]

# D and E need the same number of source ratings for a fair comparison.
number_of_ratings = len(nordic_train)
if len(non_nordic_train) < number_of_ratings:
    raise ValueError("Run E needs at least as many non-Nordic ratings as Run D.")

# The fixed seed gives us the same random sample each time.
sampled_train = non_nordic_train.sample(
    n=number_of_ratings,
    random_state=SEED,
)
number_of_users = sampled_train["user_id"].nunique()
print(f"{number_of_ratings:,} ratings from {number_of_users:,} non-Nordic users")

# Use the same Norwegian history and model size as Run D.
non_nordic_recommendations = recommend_bpr(
    sampled_train,
    history,
    factors=best_factors,
)
score_run("E", non_nordic_recommendations)

## Save results

In [ ]:
pd.DataFrame(results).to_csv("results/model_results.csv", index=False)
pd.DataFrame(results)